# Preprocessing

Demultiplexes and QC-filters one Tapestri run, then exports one H5 file per donor.

Method details and schematics: [README.md](README.md)

---

## Setup

### Load modules

Standard library, scientific stack, plotting and interactive tables, then Mission Bio Mosaic.
Empty entries are removed from `sys.path` before importing Mosaic, as they cause a `FileNotFoundError` in setuptools.

In [ ]:
import copy
import importlib
import multiprocessing
import os
import random
import sys

import itables.options as opt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.offline as pyo
import scipy.cluster.hierarchy as sch
import seaborn as sns
import yaml
from itables import init_notebook_mode
from scipy.spatial.distance import squareform

sys.path = [p for p in sys.path if p]
import missionbio.mosaic as ms

### Notebook display settings

Plotly renders offline, every DataFrame is shown as an interactive table with no row or column limits, and matplotlib plots inline.
`NUM_CORES` leaves one CPU core free.

In [ ]:
pyo.init_notebook_mode()

init_notebook_mode(all_interactive=True)
opt.lengthMenu = [5, 10, 20, 50, 100, 200, 500]
opt.maxBytes = 0
opt.maxColumns = 0

%matplotlib inline

NUM_CORES = max(1, multiprocessing.cpu_count() - 1)

### Load custom functions

The project root (`wd`) is two levels up from `notebooks/01_preprocessing/`.
All submodules are force-reloaded, so edits to the pipeline scripts are picked up without restarting the kernel.

In [ ]:
wd = os.path.dirname(os.path.dirname(os.getcwd()))

sys.path.insert(0, os.path.join(wd, "scripts"))
sys.path.insert(0, os.path.join(wd, "scripts", "pipelines"))

import DNA_Utils
import PRO_Utils
import CNV_Utils
import LOH_Pipeline

for pkg in [DNA_Utils, PRO_Utils, CNV_Utils, LOH_Pipeline]:
    for name, mod in sorted(sys.modules.items()):
        if name.startswith(pkg.__name__):
            importlib.reload(mod)

print("✓ Custom modules loaded")

### Reproducibility

Fixed seeds for Python and NumPy. `PYTHONHASHSEED` does not change hashing in the running kernel, but worker processes started later (e.g. the parallel variant filter) inherit it.

In [ ]:
%env PYTHONHASHSEED=42
random.seed(42)
np.random.seed(42)

### Notebook helpers

- `annotate_in_batches`: annotates variants through the Mosaic API, 1000 variants per request.
- `confirm_or_stop`: interactive checkpoint; stops the notebook unless the answer is `y`/`yes`.

In [ ]:
def annotate_in_batches(sample, batch_size=1000):
    n_variants = sample.dna.shape[1]
    parts = []
    for start in range(0, n_variants, batch_size):
        subset = copy.copy(sample)
        subset.dna = sample.dna[:, start:start + batch_size]
        parts.append(subset.dna.get_annotations())
    return pd.concat(parts, axis=0)


def confirm_or_stop(prompt):
    response = input(prompt).strip().lower()
    if response not in ("y", "yes"):
        raise SystemExit("Stopped by user.")

## Experiment configuration

All parameters are read from `experiment_config.yml`. Set the experiment name below:

- `EXPERIMENT`: experiment folder name under `metadata/`.
- `SKIP_CHECK`: `True` skips the interactive confirmations.
- `EXPLORE_RAW_CHR17`: `True` loads the unfiltered H5 and inspects chr17.
- `EXCLUDED_LABELS`: cell labels removed during clean-up and export.

In [ ]:
EXPERIMENT = "MDS010"

SKIP_CHECK = True
EXPLORE_RAW_CHR17 = False

EXCLUDED_LABELS = ("Mixed", "Low_quality")

### Load `experiment_config.yml`

The whitelist is a comma-separated list of variants; duplicates are dropped and the order is kept.

In [ ]:
config_path = os.path.join(wd, "metadata", EXPERIMENT, "experiment_config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

IS_MULTIPLEXED = config.get("multiplexed", False)
N_DONORS       = config.get("n_samples_multiplexed", 1)
N_CELLS        = int(config["n_cells"])
GENOME_BUILD   = config.get("genome_build", "hg19")
PROCESSING     = config.get("experimental_processing", "")
RAW_EXP_NAME   = config.get("raw_experiment_name", EXPERIMENT)

raw_wl = config.get("whitelist", "")
whitelist = list(dict.fromkeys(raw_wl.split(","))) if raw_wl else []

print(f"Experiment:    {EXPERIMENT}")
print(f"Multiplexed:   {IS_MULTIPLEXED} ({N_DONORS} donor(s))")
print(f"N_Cells:       {N_CELLS}")
print(f"Processing:    {PROCESSING}")
print(f"Genome:        {GENOME_BUILD}")
print(f"Whitelist:     {len(whitelist)} variant(s)")
for v in whitelist:
    print(f"  {v}")

### Resolve data paths

- Multiplexed runs read the H5 from `processed_data/h5/{EXPERIMENT}/multiplexed/`; single-donor runs from the first subfolder of `processed_data/h5/{EXPERIMENT}/`.
- Demultiplexing figures and tables are written to `results/demultiplexing/{EXPERIMENT}/`.

In [ ]:
h5_root = os.path.join(wd, "processed_data", "h5", EXPERIMENT)

if IS_MULTIPLEXED:
    h5_dir = os.path.join(h5_root, "multiplexed")
else:
    entries = [d for d in os.listdir(h5_root) if os.path.isdir(os.path.join(h5_root, d))]
    h5_dir = os.path.join(h5_root, entries[0]) if entries else ""

h5_files = [f for f in os.listdir(h5_dir) if f.endswith(".h5")] if os.path.isdir(h5_dir) else []
assert h5_files, f"No .h5 files found in {h5_dir}"
H5_PATH = os.path.join(h5_dir, h5_files[0])
print(f"H5 file: {H5_PATH}")

demux_dir = os.path.join(wd, "results", "demultiplexing", EXPERIMENT)
os.makedirs(demux_dir, exist_ok=True)
print(f"Demux output: {demux_dir}")

## Load data

Loads the H5 with Mission Bio's default variant filters. `min_mut_prct_cells` scales with run size: 1% of cells, clamped to 5–20 cells.
Examples and rationale: [docs §2](README.md#2-loading-data-and-the-variant-filter).

In [ ]:
min_mut_prct_cells = max(5, min(20, round(0.01 * N_CELLS))) * 100 / N_CELLS

variant_filter_config = {
    "min_dp": 5,
    "min_gq": 20,
    "vaf_ref": 5,
    "vaf_hom": 95,
    "vaf_het": 30,
    "min_prct_cells": 50,
    "min_mut_prct_cells": min_mut_prct_cells,
    "iterations": 10,
    "n_cores": NUM_CORES,
}

sample = ms.load(
    H5_PATH,
    filter_variants=True,
    filter_cells=False,
    variant_filter_config=variant_filter_config,
    whitelist=whitelist,
)

print(f"Loaded: {sample.dna.shape[0]} cells × {sample.dna.shape[1]} variants")

### Data overview

In [ ]:
for name, assay in [("DNA", sample.dna), ("Protein", sample.protein)]:
    if assay is not None:
        print(f"\n{name} assay: {assay}")
        print(f"  row_attrs: {list(assay.row_attrs.keys())}")
        print(f"  col_attrs: {list(assay.col_attrs.keys())}")
        print(f"  layers:    {list(assay.layers.keys())}")

### Explorative: raw chr17 variants (optional)

Runs only when `EXPLORE_RAW_CHR17 = True`. Loads the unfiltered H5, keeps the chr17 variants and annotates them.

In [ ]:
if EXPLORE_RAW_CHR17:
    sample_unfiltered = ms.load(H5_PATH, filter_variants=False, filter_cells=False)
    all_ids = np.asarray(sample_unfiltered.dna.ids()).astype(str)

    chr17_ids = [v for v in all_ids if v.startswith("chr17:")]
    print(f"Chr17 variants found (unfiltered): {len(chr17_ids)}")

    if not chr17_ids:
        print("No chr17 variants found.")
        chr17_annotations = pd.DataFrame()
    else:
        chr17_mask = np.isin(all_ids, chr17_ids)
        chr17_sample = copy.copy(sample_unfiltered)
        chr17_sample.dna = sample_unfiltered.dna[:, chr17_mask]

        chr17_annotations = annotate_in_batches(chr17_sample).drop_duplicates()

        display(chr17_annotations)
else:
    print("Skipped (set EXPLORE_RAW_CHR17 = True to enable)")

## Variant annotation

Batch-annotate via the Mosaic API (1000 variants per batch).

In [ ]:
variant_annotations = annotate_in_batches(sample)
print(f"Annotated {len(variant_annotations)} variants")
variant_annotations.head()

## DNA QC

### Global clustering

PCA → UMAP → graph-community clustering (multiplexed only).
The first PCA (45 components) is only for the elbow plot; the 6-component PCA is the one used for UMAP.

In [ ]:
sample_qc = copy.copy(sample)

sample_qc.dna.run_pca(components=45, attribute="AF", random_state=1337, show_plot=True)
sample_qc.dna.run_pca(components=6,  attribute="AF", random_state=1337, show_plot=False)

sample_qc.dna.run_umap(attribute="pca", random_state=1337)

if IS_MULTIPLEXED:
    sample_qc.dna.cluster(attribute="umap", method="graph-community", k=15, random_state=1337)
    print(f"Clusters: {sorted(set(sample_qc.dna.get_labels()))}")
else:
    print("Single donor — clustering skipped")

### Visualise initial clusters

In [ ]:
fig = sample_qc.dna.scatterplot(attribute="umap", colorby="label")
fig.write_image(os.path.join(demux_dir, "PCA_variants_clusters.png"))
go.Figure(fig)

### Whitelist variants in the assay

Whitelist variants missing from the filtered assay are dropped from `whitelist` here and listed in `missing_whitelist_variants.txt`.
The genotype UMAP is plotted only if at least one whitelist variant remains.

In [ ]:
assay_ids = set(map(str, sample_qc.dna.ids()))
whitelist = list(dict.fromkeys(map(str, whitelist or [])))

missing_whitelist = [v for v in whitelist if v not in assay_ids]
whitelist = [v for v in whitelist if v in assay_ids]

print(f"Whitelist kept: {len(whitelist)}")
print(f"Whitelist dropped: {len(missing_whitelist)}")
if missing_whitelist:
    print("Dropped variants (not found in assay):")
    for v in missing_whitelist:
        print(f"  - {v}")
    with open(os.path.join(demux_dir, "missing_whitelist_variants.txt"), "w") as f:
        f.write("\n".join(missing_whitelist) + "\n")

if whitelist:
    fig = sample_qc.dna.scatterplot("umap", colorby="NGT", features=whitelist)
    fig.write_image(os.path.join(demux_dir, "Distribution_TP53_variants_pre_clean_up.png"))
    display(go.Figure(fig))
else:
    print("No whitelist variants remain after filtering; plot skipped.")

### AF heatmap before clean-up

In [ ]:
fig = DNA_Utils.fast_af_heatmap(
    sample_qc,
    title="AF (pre clean-up)",
    save_path=os.path.join(demux_dir, "AF_heatmap_pre_clean_up.png"),
)

## Demultiplexing

Iteratively merges clusters with highly correlated mean AF profiles (multiplexed runs only). Flowchart and details: [docs §3](README.md#3-demultiplexing-merging-correlated-clusters).

### Merge parameters

- `MERGE_CORR_THRESHOLD`: clusters are never merged below this correlation.
- `MERGE_MARGIN`: each round merges at `max(MERGE_CORR_THRESHOLD, strongest − MERGE_MARGIN)`.
- `MAX_ITERATIONS`: upper bound on merge rounds.
- `TARGET_MAX`: stop once at most `N_DONORS + 2` clusters are left.

In [ ]:
LABEL_KEY = "label"
ATTRIBUTE = "AF"

MERGE_CORR_THRESHOLD = 0.90
MERGE_MARGIN = 0.02
MAX_ITERATIONS = 10
TARGET_MAX = N_DONORS + 2

### Merge helpers

- `_get_af_label_df`: AF matrix with the current labels in the first column.
- `_grouped_af_means`: mean AF per cluster label.
- `_cluster_corr_df`: robust cluster × cluster Pearson correlation of the grouped means (needs at least 20% of variants, minimum 5, shared).
- `_max_offdiag_corr`: highest off-diagonal correlation.

In [ ]:
def _get_af_label_df(sample, label_key=LABEL_KEY, attribute=ATTRIBUTE):
    """Extract AF matrix with current labels in the first column."""
    assay = sample.dna

    af = assay.get_attribute(attribute, constraint="row+col")
    if not isinstance(af, pd.DataFrame):
        af = pd.DataFrame(af)

    af = af.apply(pd.to_numeric, errors="coerce")

    try:
        labels = np.asarray(assay.get_labels(), dtype=object)
    except Exception:
        labels = np.asarray(assay.row_attrs[label_key], dtype=object)

    labels = pd.Series(labels, index=af.index, name=label_key).astype(str)

    return pd.concat([labels, af], axis=1)


def _grouped_af_means(df, label_key=LABEL_KEY):
    """Mean AF per current cluster label."""
    af_cols = [c for c in df.columns if c != label_key]
    grouped = (
        df.groupby(label_key, sort=True)[af_cols]
        .mean(numeric_only=True)
        .replace([np.inf, -np.inf], np.nan)
    )
    return grouped


def _cluster_corr_df(grouped_means):
    """Robust cluster × cluster Pearson correlation from grouped AF means."""
    n_variants = grouped_means.shape[1]
    min_periods = max(5, min(n_variants, int(0.20 * n_variants)))

    corr_df = grouped_means.T.corr(
        method="pearson",
        min_periods=min_periods,
    )

    corr_df = corr_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    corr_df = corr_df.clip(-1.0, 1.0)

    np.fill_diagonal(corr_df.values, 1.0)

    return corr_df


def _max_offdiag_corr(corr_df):
    """Return highest off-diagonal correlation."""
    vals = corr_df.to_numpy(dtype=float).copy()
    np.fill_diagonal(vals, -np.inf)

    if not np.isfinite(vals).any():
        return np.nan

    return float(np.nanmax(vals))

### Merge correlated clusters

Prints each merge round and shows the cluster correlation matrix after it (saved as `cluster_correlation_iteration_XX.png`), then the merge history and the updated AF heatmap (`AF_heatmap_post_merge.png`). Warns if the loop merges below `N_DONORS` clusters.

In [ ]:
if IS_MULTIPLEXED:
    df = _get_af_label_df(sample_qc)

    merge_history = []

    for iteration in range(MAX_ITERATIONS):
        grouped_means = _grouped_af_means(df)

        n_start = len(grouped_means)

        if n_start <= 1:
            print(f"All clusters merged after {iteration} iterations")
            break

        corr_df = _cluster_corr_df(grouped_means)
        max_corr = _max_offdiag_corr(corr_df)

        if not np.isfinite(max_corr):
            print(f"Stopped after {iteration + 1} iterations: no finite cluster correlations")
            break

        if max_corr < MERGE_CORR_THRESHOLD:
            print(
                f"Converged after {iteration + 1} iterations "
                f"({n_start} clusters, max off-diag corr={max_corr:.3f})"
            )
            break

        merge_corr = max(MERGE_CORR_THRESHOLD, max_corr - MERGE_MARGIN)
        merge_dist = 1.0 - merge_corr

        dist_df = 1.0 - corr_df
        dist_df = dist_df.clip(lower=0.0, upper=2.0)
        dist_df = (dist_df + dist_df.T) / 2.0
        np.fill_diagonal(dist_df.values, 0.0)

        Z = sch.linkage(
            squareform(dist_df.to_numpy(dtype=float), checks=False),
            method="average",
        )

        merged_clusters = sch.fcluster(
            Z,
            t=merge_dist,
            criterion="distance",
        )

        guide = {
            old_label: str(new_label)
            for old_label, new_label in zip(corr_df.index.astype(str), merged_clusters)
        }

        new_labels = df[LABEL_KEY].astype(str).map(guide)

        if new_labels.isna().any():
            missing = sorted(df.loc[new_labels.isna(), LABEL_KEY].astype(str).unique())
            raise ValueError(f"Missing labels in reassignment guide: {missing}")

        new_labels = new_labels.astype(str).to_numpy(dtype=object)

        sample_qc.dna.set_labels(clusters=new_labels)
        sample_qc.dna.row_attrs[LABEL_KEY] = new_labels

        df = _get_af_label_df(sample_qc)
        grouped_means = _grouped_af_means(df)
        n_clusters = len(grouped_means)

        merge_history.append({
            "iteration": iteration + 1,
            "n_clusters_before": int(n_start),
            "n_clusters_after": int(n_clusters),
            "max_corr": float(max_corr),
            "merge_corr": float(merge_corr),
            "merge_dist": float(merge_dist),
        })

        print(
            f"  Iteration {iteration + 1}: "
            f"{n_start} → {n_clusters} clusters "
            f"(merged at corr>{merge_corr:.3f}; max corr={max_corr:.3f})"
        )

        if n_clusters > 1:
            corr_df_plot = _cluster_corr_df(grouped_means)

            cg = sns.clustermap(
                corr_df_plot,
                annot=True,
                cmap="coolwarm",
                vmin=-1,
                vmax=1,
                center=0,
                annot_kws={"size": 6},
                figsize=(16, 16),
            )

            cg.ax_heatmap.set_title(
                f"Cluster correlation after merge iteration {iteration + 1} "
                f"(n={n_clusters})",
                pad=20,
            )

            corr_path = os.path.join(
                demux_dir,
                f"cluster_correlation_iteration_{iteration + 1:02d}.png",
            )
            cg.fig.savefig(corr_path, dpi=200, bbox_inches="tight")

            plt.show()

        if n_clusters <= N_DONORS:
            print(f"Warning: over-merged to {n_clusters} clusters; expected {N_DONORS}")
            break

        if n_clusters <= TARGET_MAX:
            print(f"Reached {n_clusters} clusters (target ≤{TARGET_MAX})")
            break

    merge_history = pd.DataFrame(merge_history)
    display(merge_history)

    fig = DNA_Utils.fast_af_heatmap(
        sample_qc,
        title="AF (after cluster merge)",
        save_path=os.path.join(demux_dir, "AF_heatmap_post_merge.png"),
    )

else:
    print("Single donor — demultiplexing skipped")

### Patient imputation

⚠️ **HUMAN INPUT REQUIRED** — inspect the heatmap and edit `label_map` below to map cluster labels to donor identifiers.
Single-donor runs map the default label `-` to `Donor_A`.

In [ ]:
if IS_MULTIPLEXED:
    label_map = {
        "1": "Low_quality",
        "2": "Donor_A",
        "3": "Donor_B",
        "4": "Low_quality",
        "5": "Donor_C",
    }
else:
    label_map = {
        "-": "Donor_A",
    }

sample_qc.dna.rename_labels(label_map)
print(f"Labels: {sorted(set(sample_qc.dna.get_labels()))}")

## Doublets removal

Finds cells whose AF profile mixes two donors and relabels them `Mixed` (multiplexed runs only). For each donor:
1. Each cell gets a score, `frac_changed_bidir` = 2 × min(share of variants where its AF is above the donor median, share where it is below), counting shifts of ≥ 0.30. Mosaic AF is in percent, so that is 0.3 percentage points and almost any deviation counts; in practice every variant whose donor medians differ is used (see the AF-scale note in the docs).
2. The cutoff is the elbow of the sorted scores, with a 95% bootstrap CI; one plot per donor.

Full method, worked example and caveats: [docs §4](README.md#4-doublet-removal).

In [ ]:
if IS_MULTIPLEXED:
    summary_df, donor_calls = DNA_Utils.define_doublet_cutoff_per_donor(
        sample_qc,
        label_key="label",
        exclude_labels=EXCLUDED_LABELS,
        min_frac=0.01,
        max_frac=0.45,
        slope_ratio_min=3.0,
        n_boot=300,
        ci=0.95,
        near_ci_mult=1.0,
        plot=True,
        figsize=(3.0, 2),
        plot_tiny_heatmap=True,
        heatmap_max_cells=200,
    )

    doublet_cutoffs = dict(zip(summary_df["donor"], summary_df["threshold"]))

else:
    print("Single donor — interpatient doublets non identifiable. Skipped")

### Reassign borderline cells and plot final doublet calls

Cells more than 0.02 above the cutoff become `Mixed` and cells more than 0.02 below keep their donor label. Cells in between become `Mixed` if they correlate at least as well with the `Mixed` cluster as with their donor.
Labels are written to `sample_subsets` and copied back to `sample_qc` in the next cell; an AF heatmap is shown per donor. Decision tree: [docs §4](README.md#reassign-borderline-cells-and-final-doublet-calls).

In [ ]:
if IS_MULTIPLEXED:
    donors = summary_df["donor"].tolist()

    reassign_results, final_doublet_calls, sample_subsets = DNA_Utils.apply_doublet_calls_with_corr_reassign(
        sample_qc=sample_qc,
        donor_calls=donor_calls,
        donors=donors,
        band=0.02,
        min_shared=40,
        delta_corr=0.00,
        label_key="label",
        write_back=True,
    )

    for donor in donors:
        fig = DNA_Utils.fast_af_heatmap(
            sample_subsets[donor],
            title=f"{donor}: final doublet calls",
        )

else:
    print("Single donor — interpatient doublets non identifiable. Skipped")

### Sync doublet reassignments back into `sample_qc`

Labels from each donor subset are copied to the matching barcodes in `sample_qc`.

In [ ]:
if IS_MULTIPLEXED:
    main_barcodes = np.asarray(sample_qc.dna.row_attrs["barcode"])
    main_idx = {bc: j for j, bc in enumerate(main_barcodes)}
    main_labels = np.asarray(sample_qc.dna.row_attrs["label"], dtype=object)

    for donor in donors:
        if donor not in sample_subsets:
            continue
        sub_barcodes = np.asarray(sample_subsets[donor].row_attrs["barcode"])
        sub_labels = np.asarray(sample_subsets[donor].row_attrs["label"], dtype=object)
        for i, bc in enumerate(sub_barcodes):
            j = main_idx.get(bc)
            if j is not None:
                main_labels[j] = sub_labels[i]

    sample_qc.dna.row_attrs["label"] = main_labels
    print(pd.Series(main_labels).value_counts(dropna=False))

## Low quality cell removal

Flags degraded cells per donor (amplicon dropout, noisy genotypes) and relabels them `Low_quality`; `Mixed` cells keep their label. A cell is flagged if any of three gates fires: noisy genotype at germline-stable loci, high global dropout, or high dropout and low AF together.
Global and per-donor AF heatmaps are shown and saved to `demux_dir`. Gate details, including which parameters apply to which gate: [docs §5](README.md#5-low-quality-cell-removal).

In [ ]:
summary_df_lowq, fail_lowq, scores_lowq, sample_subsets = DNA_Utils.apply_low_quality_qc(
    sample_qc=sample_qc,
    DNA_Utils=DNA_Utils,
    sample_subsets=sample_subsets if "sample_subsets" in globals() else None,
    IS_MULTIPLEXED=IS_MULTIPLEXED,
    k_mad=3.5,
    q=0.975,
    min_effect_size=2.0,
    abs_global_dropout=0.50,
    abs_low_af_info=0.50,
    depth_adaptive=True,
    abs_floor_quantile=0.95,
    use_combined_gate=True,
    combined_quantile=0.95,
    use_low_af_gate=False,
    protect_localized_loh=False,
    heatmap_save_dir=demux_dir,
)

⚠️ **CHECKPOINT** — inspect heatmaps before continuing.

In [ ]:
if not SKIP_CHECK:
    confirm_or_stop("Continue? (y/n): ")
    print("✓ Continuing")
else:
    print("⚠️  SKIP_CHECK=True — continuing")

### Propagate QC labels and filter

1. Multiplexed runs: copies the QC labels from each subset in `sample_subsets` back into `sample_qc` (requires the doublet reassignment step), then deletes `sample_subsets`.
2. Keeps only cells whose label is not in `EXCLUDED_LABELS`, subsetting the original `sample`.
3. Carries the labels and UMAP coordinates over from `sample_qc` to `filtered_sample`.

In [ ]:
main_barcodes = np.asarray(sample_qc.dna.row_attrs["barcode"]).astype(str)

if IS_MULTIPLEXED:
    if "sample_subsets" not in globals() or sample_subsets is None:
        raise RuntimeError("sample_subsets is not defined. Run the doublet reassignment step first.")

    main_idx = {bc: j for j, bc in enumerate(main_barcodes)}

    for subset in sample_subsets:
        sub_barcodes = np.asarray(sample_subsets[subset].row_attrs["barcode"]).astype(str)
        sub_labels = np.asarray(sample_subsets[subset].row_attrs["label"], dtype=object)
        for i, bc in enumerate(sub_barcodes):
            j = main_idx.get(bc)
            if j is not None:
                sample_qc.dna.row_attrs["label"][j] = sub_labels[i]

    del sample_subsets
else:
    print("Non-multiplexed run — skipping subset-label merge.")

clean_labels = [lbl for lbl in set(sample_qc.dna.row_attrs["label"]) if lbl not in EXCLUDED_LABELS]
cells_to_keep = sample_qc.dna.barcodes(clean_labels)
filtered_sample = sample[cells_to_keep]

keep_mask = np.isin(main_barcodes, np.asarray(cells_to_keep).astype(str))
filtered_sample.dna.set_labels(np.asarray(sample_qc.dna.get_labels(), dtype=object)[keep_mask])
filtered_sample.dna.row_attrs["umap"] = np.asarray(sample_qc.dna.row_attrs["umap"])[keep_mask]

removed_n = sample.dna.shape[0] - filtered_sample.dna.shape[0]
print(f"Filtered: {filtered_sample.dna.shape[0]} cells (removed {removed_n} cells labeled Mixed/Low_quality)")

### Post-filtering visualisation

In [ ]:
fig = filtered_sample.dna.scatterplot(attribute="umap", colorby="label")
fig.write_image(os.path.join(demux_dir, "PCA_variants_patients_post_clean_up.png"))
go.Figure(fig)

### Patient correlation after clean-up

Correlation of the mean AF profiles between donors (multiplexed only).

In [ ]:
if IS_MULTIPLEXED:
    assay = filtered_sample.dna
    df = assay.get_attribute("AF", constraint="row+col")
    df.loc[:, "label"] = assay.get_labels()

    grouped_means = df.groupby("label").mean()
    corr_matrix = np.corrcoef(grouped_means)

    fig = sns.clustermap(
        corr_matrix, annot=True, cmap="coolwarm",
        xticklabels=list(grouped_means.index),
        yticklabels=list(grouped_means.index),
        annot_kws={"size": 6}, figsize=(4, 4),
    )
    fig.ax_heatmap.set_title("Post-cleanup patient correlation")
    fig.savefig(os.path.join(demux_dir, "Variants_patients_correlation_matrix.png"))
    plt.show()
else:
    print("Single donor — correlation plot skipped")

### Top-variant AF heatmap

The 500 most variable variants are ordered by hierarchical clustering (average linkage on 1 − correlation, cut into as many clusters as there are donors), and the AF heatmap follows that column order.

In [ ]:
assay = filtered_sample.dna
df = assay.get_attribute("AF", constraint="row+col")
variances = df.var(axis=0)
top_cols = variances.nlargest(500).index
df_top = df.loc[:, top_cols]
dist_mat = 1 - df_top.corr()
Z = sch.linkage(dist_mat, method="average")
clusters = sch.fcluster(Z, len(set(filtered_sample.dna.row_attrs["label"])), criterion="maxclust")
ordered_cols = pd.DataFrame({"col": top_cols, "cluster": clusters}).sort_values("cluster")["col"]

fig = DNA_Utils.fast_af_heatmap(
    filtered_sample,
    label_key="label",
    attribute="AF",
    features=list(ordered_cols),
    title="Variants x patients (post clean-up)",
    width=12.0, height=16.0,
    save_path=os.path.join(demux_dir, "Variants_patients_post_clean_up.png"),
)

### Whitelist variants after clean-up

In [ ]:
fig = filtered_sample.dna.scatterplot("umap", colorby="NGT", features=whitelist)
fig.write_image(os.path.join(demux_dir, "Distribution_TP53_variants_post_clean_up.png"))
go.Figure(fig)

### Whitelist variant carriers per donor

Percentage of cells per donor carrying each whitelist variant, saved as a CSV and a heatmap.
NGT convention: 0 = WT, 1 = HET, 2 = HOM, 3 = missing; a cell counts as carrying the variant when NGT is 1 or 2.

In [ ]:
ngt = filtered_sample.dna.get_attribute("NGT", features=whitelist, constraint="row+col")
if not isinstance(ngt, pd.DataFrame):
    ngt = pd.DataFrame(ngt, columns=list(whitelist))

if ngt.index is None or len(ngt.index) != ngt.shape[0]:
    ngt.index = np.asarray(filtered_sample.dna.row_attrs["barcode"]).astype(str)

labels = pd.Series(
    np.asarray(filtered_sample.dna.row_attrs[LABEL_KEY]).astype(str),
    index=ngt.index,
    name="donor",
)

ngt_num = ngt.apply(pd.to_numeric, errors="coerce")
is_expressing = ngt_num.isin([1, 2])

pct_expr = (is_expressing.groupby(labels).mean() * 100).round(2)

pct_expr.to_csv(os.path.join(demux_dir, "whitelist_variant_pct_expressing_by_donor.csv"))

fig = px.imshow(
    pct_expr,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels=dict(x="Variant", y="Donor", color="% expressing"),
    title="% cells expressing whitelist variants by donor",
)
fig.update_layout(width=500, height=max(450, 35 * pct_expr.shape[0]))
fig.write_image(os.path.join(demux_dir, "whitelist_variant_pct_expressing_by_donor.png"))
fig.show()

### Final patient label mapping

⚠️ **HUMAN INPUT REQUIRED** — edit `final_label_map` below to map donor labels to the final patient identifiers.
Single-donor runs are already named, so the map is left empty.

In [ ]:
if not SKIP_CHECK:
    confirm_or_stop("Edit the cell below, then continue? (y/n): ")

In [ ]:
if IS_MULTIPLEXED:
    final_label_map = {
        "Donor_A": "Donor_A",
        "Donor_B": "Donor_B",
        "Donor_C": "Low_quality",
    }
else:
    final_label_map = {}

if final_label_map:
    filtered_sample.dna.rename_labels(final_label_map)

print(f"Final labels: {sorted(set(filtered_sample.dna.get_labels()))}")

## Export filtered samples

Per-donor H5 files are saved into `processed_data/h5/`: multiplexed → `{EXPERIMENT}/demultiplexed/{donor}/`, single → `{EXPERIMENT}/{donor}/`.

### Reload the raw H5 and map cells to patients

The H5 is reloaded with **all** variants (no variant filtering), so the exported files keep every raw variant.
Cell-to-patient assignment comes from `filtered_sample`; labels in `EXCLUDED_LABELS` are not exported.

In [ ]:
sample_raw = ms.load(
    H5_PATH,
    filter_variants=False,
    filter_cells=False,
)
print(f"Raw loaded: {sample_raw.dna.shape[0]} cells × {sample_raw.dna.shape[1]} variants")

if "filtered_sample" not in globals() or filtered_sample is None:
    raise RuntimeError("filtered_sample is not defined.")
if getattr(filtered_sample, "dna", None) is None:
    raise RuntimeError("filtered_sample.dna is missing.")
if getattr(sample_raw, "dna", None) is None:
    raise RuntimeError("sample_raw.dna is missing.")

barcodes = pd.Index(np.asarray(filtered_sample.dna.row_attrs["barcode"]).astype(str))
labels = pd.Series(
    np.asarray(filtered_sample.dna.get_labels()).astype(str),
    index=barcodes,
)

patient_labels = sorted(lbl for lbl in labels.unique() if lbl not in EXCLUDED_LABELS)
print(f"Exporting {len(patient_labels)} patient(s): {patient_labels}")

raw_barcodes = pd.Index(np.asarray(sample_raw.dna.row_attrs["barcode"]).astype(str))
raw_barcode_set = set(raw_barcodes)

raw_n_cells = sample_raw.dna.shape[0]
raw_n_variants = sample_raw.dna.shape[1]

### Write per-patient H5 files and results folders

For each patient:
1. Drops assigned barcodes missing from the raw H5 (with a warning).
2. Creates the per-patient results tree under `results/per_sample/{EXPERIMENT}/{patient}/`, mirroring the experiment-level layout.
3. Subsets the raw sample, checks that all raw variants are kept, and sets every DNA (and protein, if present) label to the patient ID.
4. Saves the H5, retrying once after removing any partial file.

In [ ]:
results_base = os.path.join(wd, "results", "per_sample", EXPERIMENT)

DNA_CNV_SUBDIRS = [
    "dna_cnv/variants/somatic_variants",
    "dna_cnv/variants/germline_variants",
    "dna_cnv/variants/informative_variants",
    "dna_cnv/variants/whitelist_variants",
    "dna_cnv/phylogeny",
    "dna_cnv/clonotypes/allele_frequencies",
    "dna_cnv/clonotypes/ploidy",
    "dna_cnv/matrices",
]

PROTEIN_SUBDIRS = [
    "protein/matrices",
    "protein/annotation",
]


def _sample_dims(patient_sample):
    dna_n = patient_sample.dna.shape[0]
    var_n = patient_sample.dna.shape[1]
    prot_n = (
        patient_sample.protein.shape[1]
        if getattr(patient_sample, "protein", None) is not None
        else 0
    )
    return dna_n, var_n, prot_n


for patient_id in patient_labels:
    out_name = f"{patient_id}_protein_unfiltered.dna+protein.h5"

    cell_ids = labels.index[labels == patient_id].astype(str).tolist()

    if not cell_ids:
        print(f"  ⚠ {patient_id}: no cells, skipped")
        continue

    missing_cell_ids = [bc for bc in cell_ids if bc not in raw_barcode_set]

    if missing_cell_ids:
        print(
            f"  ⚠ {patient_id}: {len(missing_cell_ids)} / {len(cell_ids)} "
            "assigned barcodes are missing from sample_raw; dropping missing cells"
        )
        cell_ids = [bc for bc in cell_ids if bc in raw_barcode_set]

    if not cell_ids:
        print(f"  ⚠ {patient_id}: no cells found in sample_raw after barcode matching, skipped")
        continue

    patient_results = os.path.join(results_base, patient_id)
    for subdir in DNA_CNV_SUBDIRS + PROTEIN_SUBDIRS:
        os.makedirs(os.path.join(patient_results, subdir), exist_ok=True)

    patient_sample = sample_raw[cell_ids]

    patient_n_cells = patient_sample.dna.shape[0]
    patient_n_variants = patient_sample.dna.shape[1]

    if patient_n_variants != raw_n_variants:
        print(
            f"  ⚠ {patient_id}: variant count mismatch after raw subsetting: "
            f"{patient_n_variants} exported vs {raw_n_variants} raw"
        )
    else:
        print(f"  ✓ {patient_id}: retained all {patient_n_variants} raw variants")

    patient_sample.dna.set_labels(
        np.array([patient_id] * patient_n_cells, dtype=object)
    )

    try:
        if getattr(patient_sample, "protein", None) is not None:
            patient_sample.protein.set_labels(
                np.array([patient_id] * patient_n_cells, dtype=object)
            )
    except Exception as e:
        print(f"  ⚠ {patient_id}: could not set protein labels — {e}")

    if IS_MULTIPLEXED:
        out_dir = os.path.join(h5_root, "demultiplexed", patient_id)
    else:
        out_dir = os.path.join(h5_root, patient_id)

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, out_name)

    try:
        ms.save(patient_sample, out_path)

        dna_n, var_n, prot_n = _sample_dims(patient_sample)
        print(
            f"  ✓ {patient_id}: {dna_n} cells, "
            f"variants={var_n}, protein_features={prot_n}"
        )
        print(f"    H5:      {out_path}")
        print(f"    Results: {patient_results}/")

    except Exception as e:
        print(f"  ⚠ {patient_id}: save failed ({e}), retrying...")

        try:
            if os.path.exists(out_path):
                os.remove(out_path)

            ms.save(patient_sample, out_path)

            dna_n, var_n, prot_n = _sample_dims(patient_sample)
            print(
                f"  ✓ {patient_id}: saved on retry — "
                f"{dna_n} cells, variants={var_n}, protein_features={prot_n}"
            )

        except Exception as e2:
            print(f"  ✗ {patient_id}: failed — {e2}")

print("\nDone.")